In [75]:
import pandas as pd
import numpy as np

In [ ]:
year=int(snakemake.wildcards.year)
cnt=snakemake.params.countries

In [77]:
summer_time_dates ={
    2010:[pd.Timestamp(2010,3,28), pd.Timestamp(2010,10,31)],
    2011:[pd.Timestamp(2011,3,27), pd.Timestamp(2011,10,30)],
    2012:[pd.Timestamp(2012,3,25), pd.Timestamp(2012,10,28)],
}

In [78]:
dates=pd.date_range(start=pd.Timestamp(year,1,1),end=pd.Timestamp(year,12,31),freq="D")

In [ ]:
utc=pd.read_csv(snakemake.input.time_zones).set_index('ISO2').loc[cnt,"UTC"]

In [ ]:
country_annual_dem_sector = pd.read_csv(snakemake.input.annual_demands, skiprows=1, index_col=0).query("index in @cnt").T.loc[:,cnt]

country_annual_dem = {}
country_annual_dem['Heavy industry'] = country_annual_dem_sector.loc['Heavy industry']
country_annual_dem['Rail'] = country_annual_dem_sector.loc['Rail']
country_annual_dem['Commercial + light industry'] = country_annual_dem_sector.loc['Commercial buildings light and appliances + Commercial other + light industry']
country_annual_dem['Residential'] = country_annual_dem_sector.loc['Residential light and appliances']

In [ ]:
# count the number weekdays, saturdays and sundays in the year.

days = dict(zip(np.unique(dates.day_name(), return_counts=True)[0],np.unique(dates.day_name(), return_counts=True)[1])) # count the number of each weekday

weekdays = days['Monday'] + days['Tuesday'] + days['Wednesday'] + days['Thursday'] + days['Friday'] # number of weekdays
saturdays = days['Saturday'] # number of saturdays
sundays = days['Sunday'] # number of sundays

In [91]:
# count the number of weekdays and weekend days (saturdays and sundays) for summer and winter. Summer and winter is assumed to follow the summer time. 

winter_days = pd.DataFrame(dates, columns=['Date']).query("Date<@summer_time_dates[@year][0] or Date>@summer_time_dates[@year][1]")
winter_days_count = dict(zip(np.unique(winter_days.Date.dt.day_name(), return_counts=True)[0],np.unique(winter_days.Date.dt.day_name(), return_counts=True)[1]))
winter_weekdays = winter_days_count['Monday'] + winter_days_count['Tuesday'] + winter_days_count['Wednesday'] + winter_days_count['Thursday'] + winter_days_count['Friday']
winter_weekend_days = winter_days_count['Saturday'] + winter_days_count['Sunday']

summer_days = pd.DataFrame(dates, columns=['Date']).query("Date>=@summer_time_dates[@year][0] and Date<=@summer_time_dates[@year][1]")
summer_days_count = dict(zip(np.unique(summer_days.Date.dt.day_name(), return_counts=True)[0],np.unique(summer_days.Date.dt.day_name(), return_counts=True)[1]))
summer_weekdays = summer_days_count['Monday'] + summer_days_count['Tuesday'] + summer_days_count['Wednesday'] + summer_days_count['Thursday'] + summer_days_count['Friday']
summer_weekend_days = summer_days_count['Saturday'] + summer_days_count['Sunday']

# Rail

In [ ]:
rail_profiles = pd.read_csv(snakemake.input.rail_profiles).set_index('Hour')

scale = (rail_profiles['Weekday'].sum()*weekdays + rail_profiles['Saturday'].sum()*saturdays + rail_profiles['Sunday'].sum()*sundays)/len(dates)
rail_profiles = rail_profiles/scale

In [ ]:
temp=pd.DataFrame(np.zeros((dates.shape[0],len(cnt)))+(1/len(dates)),columns=cnt,index=dates).mul(country_annual_dem['Rail'].values*1E6,axis=1) # distributes annual demand evenly across days

out=[]

for date,d in temp.iterrows():
     if date.day_name() in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
        prof = np.tile(rail_profiles['Weekday'].values.reshape(-1,1),reps=(1,d.shape[0]))
     elif date.day_name() == 'Saturday':
         prof = np.tile(rail_profiles['Saturday'].values.reshape(-1,1),reps=(1,d.shape[0]))
     elif date.day_name() == 'Sunday':
         prof = np.tile(rail_profiles['Sunday'].values.reshape(-1,1),reps=(1,d.shape[0]))

     if date < summer_time_dates[year][0] or date > summer_time_dates[year][1]:
          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u)
    
          out.append(d.values.reshape(1,-1)*prof)

     else:
          for i,u in enumerate(utc*-1):
            prof[:,i]=np.roll(prof[:,i],u-1)
            
          out.append(d.values.reshape(1,-1)*prof)

out=pd.DataFrame(np.concatenate(out,axis=0),
                     index=pd.date_range(start=temp.index[0],end=temp.index[-1]+pd.Timedelta(hours=23),freq="1h"),
                     columns=temp.columns)

out.to_csv(snakemake.output[0])

# Heavy industry

In [ ]:
heavy_industry_profile = pd.Series(np.ones(24)/24)

temp=pd.DataFrame(np.zeros((dates.shape[0],len(cnt)))+(1/len(dates)),columns=cnt,index=dates).mul(country_annual_dem['Heavy industry'].values*1E6,axis=1) # distributes annual demand evenly across days

out=[]

for date,d in temp.iterrows():
     prof = np.tile(heavy_industry_profile.values.reshape(-1,1),reps=(1,d.shape[0]))
     
     if date < summer_time_dates[year][0] or date > summer_time_dates[year][1]:
          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u)
    
          out.append(d.values.reshape(1,-1)*prof)

     else:
          for i,u in enumerate(utc*-1):
            prof[:,i]=np.roll(prof[:,i],u-1)
            
          out.append(d.values.reshape(1,-1)*prof)

out=pd.DataFrame(np.concatenate(out,axis=0),
                     index=pd.date_range(start=temp.index[0],end=temp.index[-1]+pd.Timedelta(hours=23),freq="1h"),
                     columns=temp.columns)

out.to_csv(snakemake.output[1])

# Residential light and appliances

In [ ]:
residential_profiles = pd.read_csv(snakemake.input.residential_light_and_appliances_profiles).set_index('Hour')
scale = (residential_profiles['SummerWeekday'].sum()*summer_weekdays + residential_profiles['SummerWeekend'].sum()*summer_weekend_days + residential_profiles['WinterWeekday'].sum()*winter_weekdays + residential_profiles['WinterWeekend'].sum()*winter_weekend_days)/len(dates)
residential_profiles = residential_profiles/scale

In [ ]:
temp=pd.DataFrame(np.zeros((dates.shape[0],len(cnt)))+(1/len(dates)),columns=cnt,index=dates).mul(country_annual_dem['Residential'].values*1E6,axis=1) # distributes annual demand evenly across days

out=[]

for date,d in temp.iterrows():
     if date < summer_time_dates[year][0] or date > summer_time_dates[year][1]:

          if date.day_name() in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
               prof = np.tile(residential_profiles['WinterWeekday'].values.reshape(-1,1),reps=(1,d.shape[0]))
          elif date.day_name() in ['Saturday', 'Sunday']:
               prof = np.tile(residential_profiles['WinterWeekend'].values.reshape(-1,1),reps=(1,d.shape[0]))

          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u)
    
          out.append(d.values.reshape(1,-1)*prof)

     else:
          
          if date.day_name() in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
               prof = np.tile(residential_profiles['SummerWeekday'].values.reshape(-1,1),reps=(1,d.shape[0]))
          elif date.day_name() in ['Saturday', 'Sunday']:
               prof = np.tile(residential_profiles['SummerWeekend'].values.reshape(-1,1),reps=(1,d.shape[0]))
          
          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u-1)
            
          out.append(d.values.reshape(1,-1)*prof)

out=pd.DataFrame(np.concatenate(out,axis=0),
                     index=pd.date_range(start=temp.index[0],end=temp.index[-1]+pd.Timedelta(hours=23),freq="1h"),
                     columns=temp.columns)

out.to_csv(snakemake.output[3])

# Commercial buildings light and appliances + Commercial other + Light industry

In [ ]:
cli_profiles = pd.read_csv(snakemake.input.commercial_and_light_industry_profiles).set_index('Hour')
scale = (cli_profiles['SummerWeekday'].sum()*summer_weekdays + cli_profiles['SummerWeekend'].sum()*summer_weekend_days + cli_profiles['WinterWeekday'].sum()*winter_weekdays + cli_profiles['WinterWeekend'].sum()*winter_weekend_days)/len(dates)
cli_profiles = cli_profiles/scale

In [ ]:
temp=pd.DataFrame(np.zeros((dates.shape[0],len(cnt)))+(1/len(dates)),columns=cnt,index=dates).mul(country_annual_dem['Commercial + light industry'].values*1E6,axis=1) # distributes annual demand evenly across days

out=[]

for date,d in temp.iterrows():
     if date < summer_time_dates[year][0] or date > summer_time_dates[year][1]:

          if date.day_name() in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
               prof = np.tile(cli_profiles['WinterWeekday'].values.reshape(-1,1),reps=(1,d.shape[0]))
          elif date.day_name() in ['Saturday', 'Sunday']:
               prof = np.tile(cli_profiles['WinterWeekend'].values.reshape(-1,1),reps=(1,d.shape[0]))

          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u)
    
          out.append(d.values.reshape(1,-1)*prof)

     else:
          
          if date.day_name() in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
               prof = np.tile(cli_profiles['SummerWeekday'].values.reshape(-1,1),reps=(1,d.shape[0]))
          elif date.day_name() in ['Saturday', 'Sunday']:
               prof = np.tile(cli_profiles['SummerWeekend'].values.reshape(-1,1),reps=(1,d.shape[0]))
          
          for i,u in enumerate(utc*-1):
               prof[:,i]=np.roll(prof[:,i],u-1)
            
          out.append(d.values.reshape(1,-1)*prof)

out=pd.DataFrame(np.concatenate(out,axis=0),
                     index=pd.date_range(start=temp.index[0],end=temp.index[-1]+pd.Timedelta(hours=23),freq="1h"),
                     columns=temp.columns)

out.to_csv(snakemake.output[2])